# 05 · Baseline TF-IDF + Regresión Logística — Grupo 17
**Paso 6 · cierre del Sprint 2 (hito S3: Modelo Base Funcional).** Entrada: `emails_v1.parquet` (`texto_modelo`).

Reporta, **medido**, lo que hasta hoy era estimación (F1 75 %, Recall 80 %):
1. Dentro de la distribución (split test): F1 macro, P/R por clase, FPR y matriz de confusión.
2. Punto de operación: umbral de phishing elegido en VALIDACIÓN para Recall ≥ 95 %, aplicado a test.
3. Sin Nigerian_Fraud (compromiso del diccionario v1).
4. **Folds fuera de distribución** (una fuente fuera a la vez + phishing 2025+): la cifra honesta.
5. Términos de mayor peso por clase (interpretabilidad).

**Decisión de Semana 3 del cronograma:** ¿F1 macro del baseline ≥ 75 %? Se responde con ambas cifras (dentro y fuera de distribución).

In [1]:
# Celda 1: Entorno
import os, sys, json, random, time
from pathlib import Path
import numpy as np, pandas as pd
SEED = 17; random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)
def detectar_entorno():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"): return "kaggle"
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        return "local"
ENV = detectar_entorno()
if ENV == "colab":
    from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/PI_G17")
elif ENV == "kaggle":
    BASE = Path("/kaggle/working/PI_G17")
else:
    BASE = Path(os.environ.get("PI_G17_BASE", Path.cwd() / "PI_G17"))
PROC, MODELS, RES = BASE / "data/processed", BASE / "models", BASE / "results"
for p in (MODELS, RES): p.mkdir(parents=True, exist_ok=True)
D = pd.read_parquet(PROC / "emails_v1.parquet")
MAX_CHARS = 5000                     # truncado declarado (DistilBERT verá aún menos: 512 tokens)
D["x"] = D["texto_modelo"].str[:MAX_CHARS]
CLASES = ["legitimo", "spam", "phishing"]; PH = 2
FOLDS = sorted(c[len("fold_"):] for c in D.columns if c.startswith("fold_"))
print(ENV, "| correos:", len(D), "| folds:", FOLDS)

Mounted at /content/drive
colab | correos: 54684 | folds: ['loso_CEAS_08', 'loso_Nigerian_Fraud', 'loso_SpamAssasin', 'loso_nazario_anual', 'loso_phishing_pot', 'temporal_phishing_2025+']


In [2]:
# Celda 2: Pipeline y utilidades de métricas
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix
def pipeline(C):
    feats = FeatureUnion([
        ("palabras", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, max_features=100_000, sublinear_tf=True)),
        ("caracteres", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=5, max_features=100_000, sublinear_tf=True))])
    return Pipeline([("tfidf", feats), ("lr", LogisticRegression(C=C, max_iter=2000, class_weight="balanced"))])
def umbral_para_recall(y, p_ph, objetivo=0.95):
    """Umbral más alto sobre P(phishing) que alcanza el recall objetivo en el conjunto dado (validación)."""
    s = np.sort(p_ph[y == PH])[::-1]
    return float(s[min(int(np.ceil(objetivo * len(s))) - 1, len(s) - 1)]) if len(s) else 0.5
def predecir(P, umbral):
    pred = P[:, :2].argmax(1); pred[P[:, PH] >= umbral] = PH; return pred
def metricas(y, pred):
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=[0, 1, 2], zero_division=0)
    no_ph = y != PH
    return {"f1_macro": round(f1_score(y, pred, average="macro", labels=[0, 1, 2], zero_division=0), 4),
            "precision": dict(zip(CLASES, np.round(p, 4).tolist())), "recall": dict(zip(CLASES, np.round(r, 4).tolist())),
            "fpr_phishing": round(float((pred[no_ph] == PH).mean()), 4) if no_ph.any() else None,
            "fpr_legitimo_a_phishing": round(float((pred[y == 0] == PH).mean()), 4) if (y == 0).any() else None,
            "n": int(len(y))}

In [ ]:
# Celda 3: Selección de C en validación y modelo principal
tr, va, te = (D["split"] == s for s in ("train", "val", "test"))
busqueda = {}
for C in (0.5, 2.0, 8.0):
    t0 = time.time(); m = pipeline(C).fit(D.loc[tr, "x"], D.loc[tr, "y"])
    busqueda[C] = {"f1_val": round(f1_score(D.loc[va, "y"], m.predict(D.loc[va, "x"]), average="macro"), 4), "s": round(time.time() - t0)}
    print(C, busqueda[C])
C_OPT = max(busqueda, key=lambda c: busqueda[c]["f1_val"])
modelo = pipeline(C_OPT).fit(D.loc[tr, "x"], D.loc[tr, "y"])
P_va, P_te = modelo.predict_proba(D.loc[va, "x"]), modelo.predict_proba(D.loc[te, "x"])
UMBRAL = umbral_para_recall(D.loc[va, "y"].values, P_va[:, PH])
y_te = D.loc[te, "y"].values
R = {"entorno": ENV, "C_optimo": C_OPT, "busqueda_C": busqueda, "umbral_phishing_val_recall95": round(UMBRAL, 4),
     "test_argmax": metricas(y_te, P_te.argmax(1)),
     "test_umbral_recall95": metricas(y_te, predecir(P_te, UMBRAL)),
     "matriz_confusion_test_umbral": confusion_matrix(y_te, predecir(P_te, UMBRAL), labels=[0, 1, 2]).tolist()}
sin_nig = (D.loc[te, "grupo"] != "Nigerian_Fraud").values
R["test_umbral_sin_nigerian"] = metricas(y_te[sin_nig], predecir(P_te[sin_nig], UMBRAL))
R["test_recall_phishing_por_grupo"] = {g: round(float((predecir(P_te, UMBRAL)[(D.loc[te, "grupo"] == g).values] == PH).mean()), 4)
                                       for g in sorted(D.loc[te & (D["clase"] == "phishing"), "grupo"].unique())}
{k: R[k] for k in ("C_optimo", "umbral_phishing_val_recall95", "test_argmax", "test_umbral_recall95")}

0.5 {'f1_val': 0.983, 's': 205}
2.0 {'f1_val': 0.9863, 's': 194}
8.0 {'f1_val': 0.9874, 's': 179}


{'C_optimo': 8.0,
 'umbral_phishing_val_recall95': 0.7932,
 'test_argmax': {'f1_macro': 0.987,
  'precision': {'legitimo': 0.9901, 'spam': 0.9934, 'phishing': 0.9753},
  'recall': {'legitimo': 0.9931, 'spam': 0.9872, 'phishing': 0.9832},
  'fpr_phishing': 0.0055,
  'fpr_legitimo_a_phishing': 0.0028,
  'n': 8203},
 'test_umbral_recall95': {'f1_macro': 0.9844,
  'precision': {'legitimo': 0.9873, 'spam': 0.9858, 'phishing': 0.9876},
  'recall': {'legitimo': 0.995, 'spam': 0.99, 'phishing': 0.961},
  'fpr_phishing': 0.0027,
  'fpr_legitimo_a_phishing': 0.0003,
  'n': 8203}}

In [ ]:
# Celda 4: Folds fuera de distribución (reentrena por fold)
R["folds"] = {}
for nombre in FOLDS:
    t0 = time.time()
    s, fuera = D[f"fold_{nombre}"], D[f"fuera_{nombre}"]
    ftr, fva, fte = (s == "train"), (s == "val"), (s == "test")
    if D.loc[ftr, "y"].nunique() < 3:
        R["folds"][nombre] = {"omitido": "train sin las 3 clases"}; continue
    m = pipeline(C_OPT).fit(D.loc[ftr, "x"], D.loc[ftr, "y"])
    u = umbral_para_recall(D.loc[fva, "y"].values, m.predict_proba(D.loc[fva, "x"])[:, PH])
    Pf = m.predict_proba(D.loc[fte, "x"]); yf = D.loc[fte, "y"].values; pred = predecir(Pf, u)
    solo_fuera = fuera[fte].values
    R["folds"][nombre] = {"test_completo": metricas(yf, pred), "solo_lo_fuera": metricas(yf[solo_fuera], pred[solo_fuera]),
                          "segundos": round(time.time() - t0)}
    print(nombre, R["folds"][nombre]["solo_lo_fuera"]["recall"], f'{R["folds"][nombre]["segundos"]}s')

loso_CEAS_08 {'legitimo': 0.937, 'spam': 0.4423, 'phishing': 0.0} 195s
loso_Nigerian_Fraud {'legitimo': 0.0, 'spam': 0.0, 'phishing': 0.4353} 210s
loso_SpamAssasin {'legitimo': 0.9645, 'spam': 0.2868, 'phishing': 0.0} 223s
loso_nazario_anual {'legitimo': 0.0, 'spam': 0.0, 'phishing': 0.8603} 221s
loso_phishing_pot {'legitimo': 0.0, 'spam': 0.0, 'phishing': 0.2766} 198s
temporal_phishing_2025+ {'legitimo': 0.0, 'spam': 0.0, 'phishing': 0.6657} 220s


In [ ]:
# Celda 5: Interpretabilidad, guardado y RESUMEN
import joblib
vocab = np.array(modelo.named_steps["tfidf"].get_feature_names_out())
coef = modelo.named_steps["lr"].coef_
R["top_terminos"] = {CLASES[k]: [t.split("__", 1)[1] for t in vocab[np.argsort(coef[k])[::-1][:15]]] for k in range(3)}
joblib.dump(modelo, MODELS / "baseline_tfidf_lr_v1.joblib")
R["decision_semana3"] = {"f1_macro_test": R["test_argmax"]["f1_macro"],
                         "f1_macro_peor_fold": min((v["test_completo"]["f1_macro"] for v in R["folds"].values() if "test_completo" in v), default=None),
                         "umbral_75": R["test_argmax"]["f1_macro"] >= 0.75}
(RES / "baseline_v1.json").write_text(json.dumps(R, indent=1, ensure_ascii=False, default=str))
print("===== RESUMEN (05) =====")
print(json.dumps(R, ensure_ascii=False, default=str, separators=(",", ":")))

===== RESUMEN (05) =====
{"entorno":"colab","C_optimo":8.0,"busqueda_C":{"0.5":{"f1_val":0.983,"s":205},"2.0":{"f1_val":0.9863,"s":194},"8.0":{"f1_val":0.9874,"s":179}},"umbral_phishing_val_recall95":0.7932,"test_argmax":{"f1_macro":0.987,"precision":{"legitimo":0.9901,"spam":0.9934,"phishing":0.9753},"recall":{"legitimo":0.9931,"spam":0.9872,"phishing":0.9832},"fpr_phishing":0.0055,"fpr_legitimo_a_phishing":0.0028,"n":8203},"test_umbral_recall95":{"f1_macro":0.9844,"precision":{"legitimo":0.9873,"spam":0.9858,"phishing":0.9876},"recall":{"legitimo":0.995,"spam":0.99,"phishing":0.961},"fpr_phishing":0.0027,"fpr_legitimo_a_phishing":0.0003,"n":8203},"matriz_confusion_test_umbral":[[3191,15,1],[18,3473,17],[23,35,1430]],"test_umbral_sin_nigerian":{"f1_macro":0.9821,"precision":{"legitimo":0.9885,"spam":0.9872,"phishing":0.9813},"recall":{"legitimo":0.995,"spam":0.99,"phishing":0.9508},"fpr_phishing":0.0027,"fpr_legitimo_a_phishing":0.0003,"n":7710},"test_recall_phishing_por_grupo":{"Nige